# Ch.2 — Classical Classifiers for Face Attributes

**FaceAI**: Compare Decision Tree, KNN, and Naive Bayes on Smiling detection.

**Goal**: Interpretable rules from trees, local similarity from KNN, probabilistic baseline from NB.

**Expected**: ~82% (Tree), ~80% (KNN), ~76% (NB) — LogReg (88%) still leads.

In [ ]:
# ── Setup & Imports ────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

IMG_DIR = Path("img")
IMG_DIR.mkdir(exist_ok=True)
SAVE_KW = dict(dpi=150, bbox_inches='tight')
np.random.seed(42)
print("Imports OK")

## §0 Data — CelebA Smiling (same as Ch.1)

In [ ]:
# ── Synthetic CelebA Proxy ─────────────────────────────
X, y = make_classification(
    n_samples=5000, n_features=200, n_informative=40,
    n_redundant=20, n_clusters_per_class=3, weights=[0.52, 0.48],
    flip_y=0.05, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## §1 Decision Tree

In [ ]:
# ── Decision Tree ──────────────────────────────────────
tree = DecisionTreeClassifier(max_depth=10, min_samples_leaf=10, random_state=42)
tree.fit(X_train, y_train)

print(f"Train accuracy: {tree.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {tree.score(X_test, y_test):.3f}")
print(f"\nTop 3 rules:")
print(export_text(tree, max_depth=3, feature_names=[f'HOG[{i}]' for i in range(200)]))

In [ ]:
# ── Tree Visualization ─────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(tree, max_depth=3, feature_names=[f'HOG[{i}]' for i in range(200)],
          class_names=['Not Smiling', 'Smiling'], filled=True, rounded=True, ax=ax,
          fontsize=8)
ax.set_title('Decision Tree for Smiling Detection (depth 3)')
fig.savefig(IMG_DIR / 'decision_tree.png', **SAVE_KW)
plt.show()

## §2 K-Nearest Neighbours

In [ ]:
# ── KNN with different k values ────────────────────────
k_values = [1, 3, 5, 7, 11, 15, 25, 51]
knn_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    score = knn.score(X_test_s, y_test)
    knn_scores.append(score)
    print(f"k={k:3d}: accuracy={score:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, knn_scores, 'bo-', linewidth=2)
ax.set_xlabel('k (number of neighbours)')
ax.set_ylabel('Test Accuracy')
ax.set_title('KNN: Accuracy vs k')
best_k = k_values[np.argmax(knn_scores)]
ax.axvline(best_k, color='red', linestyle='--', alpha=0.5, label=f'Best k={best_k}')
ax.legend()
fig.savefig(IMG_DIR / 'knn_k_sweep.png', **SAVE_KW)
plt.show()

## §3 Naive Bayes

In [ ]:
# ── Gaussian Naive Bayes ───────────────────────────────
nb = GaussianNB()
nb.fit(X_train_s, y_train)

print(f"Train accuracy: {nb.score(X_train_s, y_train):.3f}")
print(f"Test accuracy:  {nb.score(X_test_s, y_test):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, nb.predict(X_test_s),
                            target_names=['Not Smiling', 'Smiling']))

## §4 Model Comparison

In [ ]:
# ── Compare All Models ─────────────────────────────────
from sklearn.linear_model import LogisticRegression

models = {
    'LogReg (Ch.1)': LogisticRegression(C=1.0, max_iter=500, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_leaf=10, random_state=42),
    'KNN (k=7)': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes': GaussianNB(),
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    acc = model.score(X_test_s, y_test)
    if hasattr(model, 'predict_proba'):
        auc = roc_auc_score(y_test, model.predict_proba(X_test_s)[:, 1])
    else:
        auc = None
    results[name] = {'accuracy': acc, 'auc': auc}
    print(f"{name:20s}: accuracy={acc:.3f}, AUC={auc:.3f}" if auc else f"{name:20s}: accuracy={acc:.3f}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]
colors = ['#4CAF50' if n == 'LogReg (Ch.1)' else '#2196F3' for n in names]
ax.barh(names, accs, color=colors)
ax.set_xlabel('Test Accuracy')
ax.set_title('Classifier Comparison — Smiling Detection')
ax.set_xlim(0.6, 1.0)
for i, v in enumerate(accs):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center')
fig.savefig(IMG_DIR / 'model_comparison.png', **SAVE_KW)
plt.show()

## §5 Feature Importance (Tree)

In [ ]:
# ── Tree Feature Importance ────────────────────────────
importances = tree.feature_importances_
top_k = 15
top_idx = np.argsort(importances)[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(top_k), importances[top_idx], color='teal')
ax.set_yticks(range(top_k))
ax.set_yticklabels([f'HOG[{i}]' for i in top_idx])
ax.set_xlabel('Gini Importance')
ax.set_title('Top 15 Features — Decision Tree')
ax.invert_yaxis()
fig.savefig(IMG_DIR / 'tree_feature_importance.png', **SAVE_KW)
plt.show()

## §6 Tree Depth Analysis

In [ ]:
# ── Overfitting vs Depth ───────────────────────────────
depths = range(1, 30)
train_accs, test_accs = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, min_samples_leaf=5, random_state=42)
    t.fit(X_train, y_train)
    train_accs.append(t.score(X_train, y_train))
    test_accs.append(t.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(depths), train_accs, 'b-', label='Train', linewidth=2)
ax.plot(list(depths), test_accs, 'r-', label='Test', linewidth=2)
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree: Overfitting vs Depth')
ax.legend()
ax.axvline(10, color='gray', linestyle='--', alpha=0.5, label='Sweet spot')
fig.savefig(IMG_DIR / 'tree_depth_analysis.png', **SAVE_KW)
plt.show()

## §7 Summary

In [ ]:
# ── Chapter Summary ────────────────────────────────────
print("=" * 50)
print("Ch.2 — Classical Classifiers Summary")
print("=" * 50)
for name, res in results.items():
    print(f"  {name:20s}: {res['accuracy']:.3f}")
print(f"\nBest classical: {max(results, key=lambda k: results[k]['accuracy'])}")
print("\nKey insight: LogReg still leads, but trees provide interpretable rules.")
print("Constraint #4 (INTERPRETABILITY): ✅ Tree rules are human-readable")
print("Next: Ch.3 — Evaluation Metrics (is accuracy enough?)")

## Exercises

1. **Depth tuning**: Find the optimal `max_depth` that maximises test F1 (not accuracy).
2. **KNN with PCA**: Apply PCA (50 components) before KNN. Does it help accuracy and speed?
3. **Ensemble preview**: Train 10 trees with different `random_state` and average their predictions. Does the ensemble beat a single tree?

In [ ]:
# Exercise 1: Optimal depth for F1
# TODO: Sweep max_depth, compute test F1 instead of accuracy
pass

In [ ]:
# Exercise 2: KNN with PCA
# TODO: Apply PCA(n_components=50) then KNN, compare accuracy and timing
pass

In [ ]:
# Exercise 3: Manual ensemble
# TODO: Train 10 trees, average predict_proba, threshold at 0.5
pass